# Multimodal Messages

- We have seen giving text inputs to our agents, here we see how to provide text and image input to the agents and see how they respond to queries post that.
- We will encode our image and audio files into Base64 format.

## Why encode in Base64 format?

- AI APIs like Claude, GPT, Gemini etc. accept input through HTTP requests as JSON. JSON is a text based format which cannot directly have binary data (raw image bytes or audio waveforms).
- **Why Base 64 solves this?** - Base64 encoding converts binary data into ASCII text characters (A-Z, a-z, 0-9, +, /), making it JSON-safe.
    - For example: Raw image bytes: \x89PNG\r\n\x1a\n... (binary gibberish)
    - Base64: iVBORw0KGgoAAAANSUhEUgAA... (text string)

## Practical Example from my context
- When I was building an AI application that creates travel itineraries from reels sent to our service on instagram, this is where base64 encoding was used in the following manner

```py
import base64

# Read the video frame (image)
with open("reel_frame.jpg", "rb") as f:
    image_bytes = f.read()

# Encode to base64
image_b64 = base64.b64encode(image_bytes).decode('utf-8')

# Send to Gemini API
response = gemini.generate_content({
    "parts": [
        {"text": "Extract location info from this frame"},
        {"inline_data": {
            "mime_type": "image/jpeg",
            "data": image_b64  # ← Base64 string goes here
        }}
    ]
})
```

- **NOTE:** - Some modern APIs support direct binary uploads via multipart requests or pre-signed URLs, but base64 remains the standard for agent frameworks because it keeps everything in one JSON payload, which simplifies retry logic, logging, and state management in your AI pipelines.

## Text Input

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a science fiction writer, create a capital city at the users request."
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content = [{"type": "text", "text": "What is the capital of the Moon?"}
                                  ])

response = agent.invoke({
    "messages": [question]
})


print(response['messages'][-1].content)

In this sci‑fi world, the Moon’s capital is Lunopolis, also known as Selene City.

Overview
- Lunopolis sits in a sheltered crater along the Moon’s near side, where the terminator line skims the rim to give residents both long days of sunlight and dramatic twilights.
- The city is a layered mix of subsurface habitats and surface arcologies, designed to maximize radiation shielding, resource efficiency, and stunning views of Earth.

Location
- Bordering the rim of a large crater in the equatorial region; dominates the crater’s inner basin with a mesh of glassy domes and reinforced basalt towers.

Government and symbols
- The Lunar Council governs from the Crown Spire, a multipurpose citadel that houses parliament, courts, and the ceremonial office of the Moon’s Delegate.
- Emblems feature a silver crescent cradling a blue Earth, with a lattice of light threads representing solar and comms networks.

Districts and landmarks
- Crown District: The governmental core, home to the Crown Spire

## Image Input

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [9]:
print(uploader.value)

({'name': 'moon_city.png', 'type': 'image/png', 'size': 3309624, 'content': <memory at 0x00000132F9566BC0>, 'last_modified': datetime.datetime(2026, 2, 8, 1, 22, 50, 415000, tzinfo=datetime.timezone.utc)},)


In [13]:
import base64

# Get the first and only uploaded file dict

uploaded_file = uploader.value[0]

# This is a memory view
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv) # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
uploaded_file

{'name': 'moon_city.png',
 'type': 'image/png',
 'size': 3309624,
 'content': <memory at 0x00000132F9566BC0>,
 'last_modified': datetime.datetime(2026, 2, 8, 1, 22, 50, 415000, tzinfo=datetime.timezone.utc)}

In [15]:
content_mv

In [17]:
# do not print img_bytes - too long - will cause the jupyter / editor to freeze

In [18]:
img_b64

'...'

## Multimodal Question

- We pass one message where we ask the model about the image with some text asking to describe the image being passed

In [19]:
multimodal_question = HumanMessage(
    content = [
        {"type": "text", "text": "Tell me about this capital"},
        {"type": "image", "base64": img_b64, "mime_type": "image/png"
         }]
)

response = agent.invoke(
    {"messages": [multimodal_question]}
)

In [20]:
print(response['messages'][-1].content)

Meet Frostspire Prime, the capital you’re looking at in the image. It sits in a frost-wedged valley carved into jagged mountains, on a world whose skies host two giant celestial neighbors—a pale gas giant hanging on the horizon and a cratered moon tugging at the night. The city is a human-made cathedral to resilience, engineering, and the art of living in extreme cold.

What Frostspire Prime is like

- Architecture and layout
  - The city is a vast labyrinth of arcologies—rings stacked like ice-sculpted pancakes atop towering rock outcrops. Each ring is a self-contained habitat: living quarters, markets, schools, clinics, and workshops, connected by magnetic elevators and sheltered skybridges.
  - At the heart stands a colossal central sphere, the Heart of Frostspire. It’s a geometic, patchwork dome that looks as if it’s been rebuilt piece by piece over centuries. The Sphere houses the government, the great archive, and the ceremonial spaces where the city’s future is debated.
  - Surr

# Audio Input

- Same Principle, we get a base64 encoding of an mp3 or .wav file, pass it to the agent along with a text question, we will use the libraries that allow us to record straight into the notebook

In [23]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm  # ← Also fix import (use 'from')

# Recording Settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

# Progress bar for the duration
for _ in tqdm(range(duration * 10)):  # ✅ Fixed
    time.sleep(0.1)

sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  9.89it/s]

Done.


In [29]:
agent = create_agent(
    model="gpt-4o-audio-preview",
)

multimodal_question = HumanMessage(content = [
    {"type":"text", "text": "Tell me about this audio file"},
    {"type":"audio", "base64": aud_b64, "mime_type":"audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

In white they march across the field,
A glorious history, a shield,
Champions born from dreams and sweat,
Real Madrid, the name they set.

From Santiago Bernabéu’s grand design,
Legends forged in the passing time,
Di Stéfano, Zidane, and more,
Echoing through each goal they score.

The Bernabéu roars with pride,
With every ball that’s struck inside,
The passion flows like rivers wide,
Beneath the Madridista tide.

Thirteen times they’ve touched the stars,
In Europe’s realm, they’ve set the bars,
The crown they wear, both old and new,
In white, the color of the true.

Real Madrid, the kings of lore,
Their anthem rings, the spirit soars,
In every heart, their story lives,
A club that dreams, a club that gives.


In [ ]:
# EOF